# MPI-Parallel Heat Simulator

### 1. Initialize and Finalize MPI

We initialize MPI right at the start of our application, and finalize it at the end.

```cpp
int main(int argc, char *argv[]) {
    MPI_Init(&argc, &argv);

    // ...

    MPI_Finalize();

    return 0;
}
```


### 2. Query MPI Information

In the initialization phase, we query the total number of ranks and ID of the currently executing rank.

```cpp
int numRanks = 0;
MPI_Comm_size(MPI_COMM_WORLD, &numRanks);

int rank = 0;
MPI_Comm_rank(MPI_COMM_WORLD, &rank);
```


### 3. Map MPI Ranks to GPUs

In the most basic case, each MPI rank is responsible for exactly one GPU.
We also assume a constant number of GPUs per node to facilitate our implementation (otherwise additional information exchange and/ or custom communicators are required).

```cpp
int numDevicesPerNode = 0;
checkCudaError(cudaGetDeviceCount(&numDevicesPerNode));

int deviceId = rank % numDevicesPerNode;
checkCudaError(cudaSetDevice(deviceId));
```

This should be done **before** any device allocations or kernel launches - usually in the setup phase of the application.

Note that some combinations of batch system and MPI implementation will set the environment variable `CUDA_VISIBLE_DEVICES` for each of the ranks spawned.
This will take care that each rank can only see a subset of the node-local GPUs.

### 4. Allocate Patch-Specific Host and Device Memory

Each rank owns **only its patch** - there is no global host allocation any more.
Instead, we allocate host and device memory sized to the local patch (including halos).
As such, it is convenient to pull the host pointer into the `Patch` struct which we keep for now.

```cpp
struct Patch {
    // ... as before

    // pointers to CPU allocation
    double* localU;
    double* localUNew;

    // pointers to the GPU allocation
    double* d_localU;
    double* d_localUNew;

    // ... as before
};
```


The allocation is then simplified in the setup phase.

```cpp
// allocate CPU
checkCudaError(cudaMallocHost(&patch.localU, patch.localSize));
checkCudaError(cudaMallocHost(&patch.localUNew, patch.localSize));

// allocate GPU
checkCudaError(cudaMalloc((void **)&patch.d_localU, patch.localSize));
checkCudaError(cudaMalloc((void **)&patch.d_localUNew, patch.localSize));
```


### 5. Switch to Distributed Initialization

The previously used `initTemperature` function assumes initialization of one big global array.
To match our partitioned host allocations, we switch to `initTemperaturePatch` instead.
This function implements the same logic as before, but receives additional **global offsets** of the local patch to apply initial conditions (e.g., a heat pulse in global coordinates) correctly.

```cpp
initTemperaturePatch(patch.localU, patch.localUNew,
    globalNumCellsX, globalNumCellsY,                   // global index space
    patch.localNumCellsX, patch.localNumCellsY,         // local index space
    patch.globalInnerBeginX, patch.globalInnerBeginY    // global offset for this patch
);
```

Note that we assume that the initial heat pulse has no effect on halos - if this is not the case, an additional communication step (see below) is necessary after initialization

Copying the data to the device is now also streamline compared to previous versions.

```cpp
// copy data to GPU
checkCudaError(cudaMemcpy(patch.d_localU, patch.localU, patch.localSize, cudaMemcpyHostToDevice));

// initialize uNew by copying from u
checkCudaError(cudaMemcpy(patch.d_localUNew, patch.d_localU, patch.localSize, cudaMemcpyDeviceToDevice));
```


### 6. Implement Halo Exchange

We use non-blocking, pairwise exchanges with neighbors.
Using CUDA-aware MPI allows us passing device pointers to MPI directly, as discussed in the previous section.

We start by preparing four requests (one for sending and one for receiving, for each of the two neighbors).
The requests are initialized with `MPI_REQUEST_NULL` to allow waiting for them even if no communication took place, e.g. when the first patch/ rank would communicate to its bottom neighbor.

```cpp
MPI_Request requests[4] = { MPI_REQUEST_NULL, MPI_REQUEST_NULL, MPI_REQUEST_NULL, MPI_REQUEST_NULL };
```


Then we check whether there is a bottom neighbor and, if so, we initiate the exchange.

```cpp
// exchange data with lower neighbor
if (rank > 0) {
    auto recvPtr = &patch.d_localUNew[0 * patch.localNumCellsX];    // bottom halo of current patch
    auto sendPtr = &patch.d_localUNew[1 * patch.localNumCellsX];    // first inner row of current patch
    MPI_Irecv(
        recvPtr,                // destination pointer
        patch.localNumCellsX,   // number of elements: one row
        MPI_DOUBLE,             // datatype
        rank - 1,               // source rank
        0,                      // tag
        MPI_COMM_WORLD,         // communicator
        &requests[0]);          // request handle - req for wait later
    MPI_Isend(
        sendPtr,                // source pointer
        patch.localNumCellsX,   // number of elements: one row
        MPI_DOUBLE,             // datatype
        rank - 1,               // destination rank
        0,                      // tag
        MPI_COMM_WORLD,         // communicator
        &requests[1]);          // request handle - req for wait later
}
```


We perform the same for the top neighbor - if it exists.

```cpp
// exchange data with upper neighbor
if (rank < numRanks - 1) {
    auto recvPtr = &patch.d_localUNew[(patch.localNumCellsY - 1) * patch.localNumCellsX];   // top halo of current patch
    auto sendPtr = &patch.d_localUNew[(patch.localNumCellsY - 2) * patch.localNumCellsX];   // last inner row of current patch
    MPI_Irecv(
        recvPtr,                // destination pointer
        patch.localNumCellsX,   // number of elements: one row
        MPI_DOUBLE,             // datatype
        rank + 1,               // source rank
        0,                      // tag
        MPI_COMM_WORLD,         // communicator
        &requests[2]);          // request handle - req for wait later
    MPI_Isend(
        sendPtr,                // source pointer
        patch.localNumCellsX,   // number of elements: one row
        MPI_DOUBLE,             // datatype
        rank + 1,               // destination rank
        0,                      // tag
        MPI_COMM_WORLD,         // communicator
        &requests[3]);          // request handle - req for wait later
}
```


The last step is waiting for the asynchronous operations to finish.
For convenience, we ignore the reported statuses here.
In practice, checking them makes the code more robust.

```cpp
// wait for all communications to complete
MPI_Waitall(4, requests, MPI_STATUSES_IGNORE);
```


### 7. Update File Output

We re-enable output with a simple, deterministic order: each rank copies from device to host (in parallel), then ranks print **sequentially**.
This is straight-forward and robust, but limits performance and scalability.
Optimization could be done by
* writing one distinct file per rank and concatenating them in a post-processing step (not recommended), 
* using parallel file I/O libraries, or
* using MPI-IO.

```cpp
    // all ranks copy data back to CPU in parallel
    checkCudaError(cudaMemcpy(patch.localU, patch.d_localU, patch.localSize, cudaMemcpyDeviceToHost));

    // Note: brute-force full-synchronize approach for simplicity - optimize with point-to-point messages triggering next write
    for (int printRank = 0; printRank < numRanks; ++printRank) {
        MPI_Barrier(MPI_COMM_WORLD);    // make sure all ranks are in the same iteration

        if (rank == printRank) {        // only one rank per loop iteration is allowed to write
            writeTemperaturePatchNpy("../output/temperature_" + idx + ".npy",
                patch.localU,
                globalNumCellsY, globalNumCellsX, patch.localNumCellsX, patch.localNumCellsY,
                numPatches, rank);
        }
    }
```


### 8. Implement Temperature Reduction

In our current implementation, each rank computes the aggregated temperature for its patch.
These partial sums can now be reduced using MPI.
For simplicity, we will use a **host** reduction for now.

```cpp
auto rankTotalTemperature = accumulateTemperature(u, patch.localNx, patch.localNy);
std::cout << "  Total temperature on rank " << rank << " is " << rankTotalTemperature << std::endl;

double totalTemperature = 0.0;
MPI_Reduce(
    &rankTotalTemperature,  // send buffer
    &totalTemperature,      // receive buffer
    1,                      // count
    MPI_DOUBLE,             // datatype
    MPI_SUM,                // operation
    0,                      // root
    MPI_COMM_WORLD);        // communicator

if (0 == rank)              // print only on one rank
    std::cout << "  Total temperature is " << totalTemperature << std::endl;
```


## Exercise

Choose one of the difficulty levels below to tailor the exercise to your preferences.
Each level provides a different starting point implementation to be copied into [stencil-2d.cu](../src/11-mpi/stencil-2d.cu).
Use it for your implementation and follow the steps outlined above.

Below the difficulty level descriptions, there are cells for compiling, executing and profiling the your solution.

### Level Hard

Create and empty file [stencil-2d.cu](../src/11-mpi/stencil-2d.cu) and copy one of the previous code versions into it (your work or a solution).

In [ ]:
!touch ../src/11-mpi/stencil-2d.cu

### Level Medium

[stencil-2d-medium.cu](../src/11-mpi/stencil-2d-medium.cu) contains a partial solution with TODOs ranging from straight-forward to complex.
Copy the provided code into the working file [stencil-2d.cu](../src/11-mpi/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
!cp ../src/11-mpi/stencil-2d-medium.cu ../src/11-mpi/stencil-2d.cu

### Level Easier

[stencil-2d-easier.cu](../src/11-mpi/stencil-2d-easier.cu) contains a partial solution with TODOs.
The solution is further progressed than the level medium version, and the complexity of the TODOs is limited.
Copy the provided code into the working file [stencil-2d.cu](../src/11-mpi/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
!cp ../src/11-mpi/stencil-2d-easier.cu ../src/11-mpi/stencil-2d.cu

### Possible Solution

[stencil-2d-solution.cu](../src/11-mpi/stencil-2d-solution.cu) contains a possible solution for this exercise.
Copy it into the working file [stencil-2d.cu](../src/11-mpi/stencil-2d.cu) with the cell below.

In [ ]:
!cp ../src/11-mpi/stencil-2d-solution.cu ../src/11-mpi/stencil-2d.cu

### Compilation, Execution and Profiling

The new code version is available in [11-mpi/stencil-2d.cu](../src/11-mpi/stencil-2d.cu) (after creating it with one of the commands above).
It can be compiled and executed with the following cells.

In [ ]:
!nvcc -ccbin=mpic++ -O3 -gencode arch=compute_80,code=sm_80 -gencode arch=compute_86,code=sm_86 -o ../build/11-mpi ../src/11-mpi/stencil-2d.cu

The next cell produces output for a small grid.
Visualize the output using the [visualize](./99-visualize.ipynb) notebook after executing the application.

By default, it runs 2 ranks on a single GPU.
Use this to verify that your application works as intended.

In [ ]:
!mpirun -n 2 ../build/11-mpi 256 64 2 2000 100

The next cell produces no output and runs a larger grid.
Use it for performance evaluation.

In [ ]:
!mpirun -n 2 ../build/11-mpi $((32 * 1024)) 256 2 8 0

Instead of running on the local **single A40** GPU, we can also submit a batch job running on **multiple A100** GPUs.
Feel free to tune the number of GPUs (both in the `--gres=gpu:a100:NGPU` and in the `mpirun -n NGPU`) to anything between one and eight GPUs.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/11-mpi.out --error=../output/11-mpi.err \
    --wrap="mpirun -n 2 ../build/11-mpi $((32 * 1024)) 256 2 8 0"

cat ../output/11-mpi.out

The next cell performs profiling with Nsight Systems by submitting a batch job.
Feel free to tune the number of GPUs (both in the `--gres=gpu:a100:NGPU` and in the `mpirun -n NGPU`) to anything between one and eight GPUs.

The profile is then available at [profiles/11-mpi.nsys-rep](../profiles/11-mpi.nsys-rep) and can be downloaded by **shift + right-clicking** the link, by clicking the link with the **middle mouse button**, or using the JupyterHub file tree.

After downloading it, open it up locally to visualize the run-time behavior of your application.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:2 \
    --time 00:05:00 --wait \
    --output=../output/11-mpi-nsys.out --error=../output/11-mpi-nsys.err \
    --wrap="nsys profile --stats=true --force-overwrite=true \
        -o ../profiles/11-mpi \
        mpirun -n 2 \
            ../build/11-mpi $((32 * 1024)) 256 2 8 0"

cat ../output/11-mpi-nsys.out

## Next Step

The next step is applying our previously discussed optimization of overlapping data transfers with computation to our MPI-parallel application in [12](./12-overlap.ipynb).